# TriageGuard: Multi-Modal ESI Prediction with NLP, Fairness-Constrained Learning, and a Real-Time Clinical Decision Support Interface

**A production-grade, clinically-grounded AI triage assistant**

---

## Overview

This notebook presents a **three-layer architecture** for AI-assisted emergency triage:

| Layer | What it does | Why it matters |
|---|---|---|
| **Layer 1 — Structured ML** | Stacked ensemble (LightGBM + XGBoost + Ordinal LR) on vitals, demographics, and derived clinical features | High-accuracy baseline; ordinal calibration reduces dangerous mis-classification |
| **Layer 2 — Free-text NLP** | TF-IDF + clinical keyword extraction on raw chief complaint text; late-fused with structured model | Chief complaints carry signal the structured data misses; this is how real triage works |
| **Layer 3 — Fairness-constrained re-ranking** | Post-hoc equalized odds correction that adjusts predictions for statistically undertriaged subgroups without requiring retraining | Addresses the core patient-safety problem: bias is in the training signal, not just the features |

**Data source:** Triagegeist competition files (`train.csv`, `test.csv`, `chief_complaints.csv`, `patient_history.csv`) — 80,000 training visits with 40 features plus free-text complaints and 25 comorbidity flags.

**Core clinical motivation:** ESI inter-rater reliability in the literature sits at κ ≈ 0.30–0.50. Our goal is not to replace the triage nurse — it is to surface cases where AI disagrees meaningfully, especially in patient groups where bias has been documented: Black and Hispanic patients (Blankenship et al., 2020), women presenting with chest pain (Rathore et al., 2000), elderly patients (Platts-Mills et al., 2012), and psychiatric presentations (Wahlbeck et al., 2011).

**Novel contributions:**
1. Late fusion of NLP chief-complaint embeddings with structured vitals — underexplored in triage AI
2. Post-hoc equalized-odds correction as a deployable fairness layer (no retraining needed at inference time)
3. Calibrated undertriage risk score (URS) per patient — a single actionable number for the clinician
4. A working clinical decision support prototype demonstrating the full patient intake → alert pipeline


## 1. Environment Setup

In [ ]:
# ── Core ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings, re, os, json
warnings.filterwarnings('ignore')

# ── ML ────────────────────────────────────────────────────────────────────────
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_predict
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    classification_report, confusion_matrix, cohen_kappa_score,
    roc_auc_score, brier_score_loss, accuracy_score, f1_score
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import StackingClassifier
from scipy import stats
from scipy.special import softmax

import xgboost as xgb
import lightgbm as lgb

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Plotting style ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 120
})
ESI_COLORS = ['#c62828', '#e65100', '#f9a825', '#2e7d32', '#1565c0']

print('Environment ready.')
print(f'NumPy {np.__version__} | Pandas {pd.__version__}')

## 2. Data Loading

We load all four provided competition files and join them on `patient_id`. The NOTE.md confirms all data is synthetic but calibrated to published MIMIC-IV-ED / NHAMCS distributions.

| File | Rows | Key contents |
|---|---|---|
| `train.csv` | 80,000 | Structured features + target `triage_acuity` |
| `test.csv` | 20,000 | Structured features (no target) |
| `chief_complaints.csv` | 100,000 | Free-text complaint; joins on `patient_id` |
| `patient_history.csv` | 100,000 | 25 binary `hx_` comorbidity flags |

In [ ]:
BASE = '/kaggle/input/competitions/triagegeist'
if not os.path.exists(BASE):
    BASE = '.'

train_raw = pd.read_csv(f'{BASE}/train.csv')
test_raw  = pd.read_csv(f'{BASE}/test.csv')
cc        = pd.read_csv(f'{BASE}/chief_complaints.csv')
hx        = pd.read_csv(f'{BASE}/patient_history.csv')
sample    = pd.read_csv(f'{BASE}/sample_submission.csv')

print('Loaded files:')
print(f'  train_raw: {train_raw.shape}')
print(f'  test_raw : {test_raw.shape}')
print(f'  cc       : {cc.shape}')
print(f'  hx       : {hx.shape}')
print(f'  sample   : {sample.shape}')

In [ ]:
# ── Merge all tables on patient_id ────────────────────────────────────────────
candidate_cols = [
    'chief_complaint_raw',
    'complaint_text',
    'chief_complaint_text',
    'chief_complaint',
    'complaint'
]

cc_col = next((c for c in candidate_cols if c in cc.columns), None)
if cc_col is None:
    text_cols = [c for c in cc.columns if c != 'patient_id' and cc[c].dtype == object]
    if not text_cols:
        raise ValueError("No complaint text column found in chief_complaints.csv")
    cc_col = text_cols[0]

cc = cc.rename(columns={cc_col: 'complaint_text'})

cc_dedup = (
    cc.sort_values('patient_id')
      .groupby('patient_id', as_index=False)['complaint_text']
      .first()
)

hx_dedup = (
    hx.drop_duplicates('patient_id')
    if 'patient_id' in hx.columns
    else hx.reset_index().rename(columns={'index': 'patient_id'})
)

def merge_all(base_df):
    df = base_df.merge(cc_dedup, on='patient_id', how='left')
    df = df.merge(hx_dedup, on='patient_id', how='left')
    if 'complaint_text' not in df.columns:
        df['complaint_text'] = ''
    df['complaint_text'] = df['complaint_text'].fillna('').astype(str)
    return df

train = merge_all(train_raw)
test  = merge_all(test_raw)

print(f'Merged train: {train.shape}')
print(f'Merged test : {test.shape}')
print(f'Complaint text column used: {cc_col}')
print(f'\nMissingness (train, top 10):')
miss = train.isnull().sum().sort_values(ascending=False)
print(miss[miss > 0].head(10).to_string())

## 3. Exploratory Data Analysis

Before modeling we audit the data for:
- Class imbalance (affects recall on high-acuity, safety-critical ESI 1–2)
- Vital sign distributions by acuity level
- Demographic composition and complaint frequency
- Missingness patterns (NOTE.md warns these are clinically non-random)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle('Triagegeist EDA — Structured + Demographic Overview', fontsize=14, fontweight='bold', y=1.01)

# 1. ESI distribution
ax = axes[0, 0]
esi_counts = train['triage_acuity'].value_counts().sort_index()
bars = ax.bar(['ESI 1\nCritical','ESI 2\nEmergent','ESI 3\nUrgent','ESI 4\nLess Urgent','ESI 5\nNon-Urgent'],
              esi_counts.values, color=ESI_COLORS, edgecolor='white', linewidth=0.7)
for bar, cnt in zip(bars, esi_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{cnt/len(train)*100:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.set_title('ESI Level Distribution', fontweight='bold')
ax.set_ylabel('Patient Count')

# 2. Age by ESI
ax = axes[0, 1]
for esi, col in zip([1,2,3,4,5], ESI_COLORS):
    ax.hist(train[train['triage_acuity']==esi]['age'], bins=20, alpha=0.5,
            color=col, label=f'ESI {esi}', density=True)
ax.set_title('Age Distribution by ESI Level', fontweight='bold')
ax.set_xlabel('Age (years)')
ax.legend(fontsize=8)

# 3. Vital sign heatmap by ESI
ax = axes[0, 2]
vital_cols = ['heart_rate', 'systolic_bp', 'spo2', 'respiratory_rate', 'temperature_c']
available_vitals = [c for c in vital_cols if c in train.columns]
vitals_esi = train.groupby('triage_acuity')[available_vitals].mean()
vitals_norm = (vitals_esi - vitals_esi.mean()) / (vitals_esi.std() + 1e-8)
sns.heatmap(vitals_norm.T, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=ax, linewidths=0.5,
            xticklabels=[f'ESI {i}' for i in range(1,6)])
ax.set_title('Vital Signs Z-Scores by ESI\n(red=abnormal, green=normal)', fontweight='bold')

# 4. Chief complaint system distribution
ax = axes[1, 0]
if 'chief_complaint_system' in train.columns:
    cc_esi = train.groupby('chief_complaint_system')['triage_acuity'].apply(
        lambda x: (x <= 2).mean()).sort_values(ascending=True)
    cc_esi.plot(kind='barh', ax=ax, color='#d32f2f', alpha=0.8)
    ax.axvline(train['triage_acuity'].le(2).mean(), ls='--', color='navy', label='Overall avg')
    ax.set_title('% High-Acuity (ESI≤2) by\nComplaint System', fontweight='bold')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'chief_complaint_system\nnot present', ha='center', va='center', transform=ax.transAxes)

# 5. Sex breakdown
ax = axes[1, 1]
if 'sex' in train.columns:
    sex_esi = train.groupby(['sex','triage_acuity']).size().unstack(fill_value=0)
    sex_esi_pct = sex_esi.div(sex_esi.sum(axis=1), axis=0)
    sex_esi_pct.plot(kind='bar', stacked=True, ax=ax, color=ESI_COLORS, edgecolor='white', linewidth=0.5)
    ax.set_title('ESI Distribution by Sex', fontweight='bold')
    ax.legend(title='ESI', bbox_to_anchor=(1,1), fontsize=8)
    ax.tick_params(axis='x', rotation=0)
else:
    ax.text(0.5, 0.5, 'sex not present', ha='center', va='center', transform=ax.transAxes)

# 6. Missingness pattern
ax = axes[1, 2]
miss_pct = train.isnull().mean().sort_values(ascending=False).head(12)
miss_pct.plot(kind='barh', ax=ax, color='#7b1fa2', alpha=0.8)
ax.set_title('Missingness Rate (Top 12 Cols)', fontweight='bold')
ax.set_xlabel('Fraction Missing')
ax.axvline(0.05, ls='--', color='gray', alpha=0.6, label='5% threshold')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA complete.')

## 4. Feature Engineering — Structured Data

We construct features in four groups:

1. **Vital sign abnormality flags** — binary indicators for clinical thresholds (tachycardia, hypotension, hypoxia, etc.) based on standard reference ranges
2. **Composite clinical scores** — shock index, NEWS2-inspired aggregate abnormality count, pulse pressure
3. **Age / temporal / demographic features** — pediatric and geriatric flags, arrival hour, shift, season
4. **Comorbidity burden** — sum of `hx_` flags; individual high-risk flags (malignancy, heart failure, COPD)

In [ ]:
HX_COLS = [c for c in train.columns if c.startswith('hx_')]

def engineer_features(df):
    d = df.copy()

    # ── Vital sign abnormality flags ──────────────────────────────────────────
    if 'heart_rate' in d:
        d['hr_tachy']     = (d['heart_rate'] > 100).astype(float)
        d['hr_brady']     = (d['heart_rate'] < 60).astype(float)
    if 'systolic_bp' in d:
        d['hypotension']  = (d['systolic_bp'] < 90).astype(float)
        d['hypertension'] = (d['systolic_bp'] > 180).astype(float)
    if 'spo2' in d:
        d['hypoxia']      = (d['spo2'] < 94).astype(float)
        d['severe_hypoxia'] = (d['spo2'] < 90).astype(float)
    if 'temperature_c' in d:
        d['fever']        = (d['temperature_c'] > 38.3).astype(float)
        d['hypothermia']  = (d['temperature_c'] < 36.0).astype(float)
    if 'respiratory_rate' in d:
        d['tachypnea']    = (d['respiratory_rate'] > 20).astype(float)
        d['bradypnea']    = (d['respiratory_rate'] < 10).astype(float)

    # ── Composite scores ──────────────────────────────────────────────────────
    if 'heart_rate' in d and 'systolic_bp' in d:
        d['shock_index'] = d['heart_rate'] / (d['systolic_bp'].clip(1) + 1e-6)
        d['shock_flag']  = (d['shock_index'] > 1.0).astype(float)

    if 'systolic_bp' in d and 'diastolic_bp' in d:
        d['pulse_pressure'] = d['systolic_bp'] - d['diastolic_bp']
        d['pp_narrow']      = (d['pulse_pressure'] < 25).astype(float)
        d['pp_wide']        = (d['pulse_pressure'] > 60).astype(float)

    # NEWS2-inspired abnormality count
    abnorm_cols = [c for c in ['hr_tachy','hr_brady','hypotension','severe_hypoxia',
                               'fever','tachypnea','shock_flag'] if c in d.columns]
    if abnorm_cols:
        d['news2_proxy'] = d[abnorm_cols].fillna(0).sum(axis=1)
        d['news2_high']  = (d['news2_proxy'] >= 3).astype(float)  # RED score threshold

    # ── GCS / pain score special handling ─────────────────────────────────────
    if 'pain_score' in d:
        d['pain_missing'] = (d['pain_score'] == -1).astype(float)  # -1 = not recorded
        d['pain_score_clean'] = d['pain_score'].replace(-1, np.nan)
        d['pain_severe']  = (d['pain_score_clean'] >= 8).astype(float)

    if 'gcs_total' in d:
        d['gcs_altered']  = (d['gcs_total'] < 15).astype(float)
        d['gcs_critical'] = (d['gcs_total'] <= 8).astype(float)

    # ── Age features ──────────────────────────────────────────────────────────
    if 'age' in d:
        d['is_infant']       = (d['age'] < 2).astype(float)
        d['is_pediatric']    = (d['age'] < 18).astype(float)
        d['is_elderly']      = (d['age'] >= 65).astype(float)
        d['is_very_elderly'] = (d['age'] >= 75).astype(float)
        d['age_sq']          = d['age'] ** 2  # capture non-linearity

    # ── Arrival context ───────────────────────────────────────────────────────
    if 'arrival_mode' in d:
        d['by_ambulance'] = (d['arrival_mode'].astype(str).str.lower().str.contains('ambul')).astype(float)

    if 'arrival_hour' in d:
        # Overnight 10pm–6am: evidence of different staffing/acuity mix
        d['overnight'] = ((d['arrival_hour'] >= 22) | (d['arrival_hour'] < 6)).astype(float)
        # Peak hours 10am–4pm
        d['peak_hours'] = ((d['arrival_hour'] >= 10) & (d['arrival_hour'] < 16)).astype(float)

    # ── Comorbidity burden ────────────────────────────────────────────────────
    if HX_COLS:
        d['comorbidity_count'] = d[HX_COLS].fillna(0).sum(axis=1)
        d['multi_comorbid']    = (d['comorbidity_count'] >= 3).astype(float)
        # High-risk individual flags
        for risky in ['hx_malignancy', 'hx_heart_failure', 'hx_copd', 'hx_renal_failure',
                      'hx_dementia', 'hx_liver_disease', 'hx_immunocompromised']:
            if risky in d.columns:
                d[risky] = d[risky].fillna(0)

    # ── High-risk complaint flag ───────────────────────────────────────────────
    HIGH_RISK = {'chest pain', 'dyspnea', 'sob', 'shortness of breath',
                 'altered mental status', 'syncope', 'stroke', 'seizure',
                 'cardiac arrest', 'respiratory distress', 'trauma'}
    if 'chief_complaint_system' in d.columns:
        d['high_risk_system'] = d['chief_complaint_system'].astype(str).str.lower().apply(
            lambda x: int(any(h in x for h in HIGH_RISK))
        )

    # ── Encode categoricals ────────────────────────────────────────────────────
    cat_cols = [c for c in ['sex', 'arrival_mode', 'shift', 'arrival_season',
                             'insurance_type', 'language', 'transport_origin',
                             'mental_status_triage', 'chief_complaint_system',
                             'age_group', 'pain_location'] if c in d.columns]
    d = pd.get_dummies(d, columns=cat_cols, drop_first=False, dtype=float)

    return d


train_feat = engineer_features(train)
test_feat  = engineer_features(test)

# Align columns (test may have fewer dummies if some categories don't appear)
SKIP_COLS = ['patient_id', 'triage_acuity', 'complaint_text',
             'disposition', 'ed_los_hours']  # outcome cols — drop from features
SKIP_COLS += [c for c in train_feat.columns if train_feat[c].dtype == object]

FEATURE_COLS = [c for c in train_feat.columns if c not in SKIP_COLS]
# Align test to same columns
for c in FEATURE_COLS:
    if c not in test_feat.columns:
        test_feat[c] = 0

X_struct = train_feat[FEATURE_COLS].copy()
y        = train_feat['triage_acuity'].copy()
X_test_struct = test_feat[FEATURE_COLS].copy()

print(f'Feature matrix: {X_struct.shape}')
print(f'Target dist:\n{y.value_counts().sort_index().to_string()}')

## 5. NLP Layer — Chief Complaint Text Embeddings

The `chief_complaints.csv` file provides raw free-text entries like *"chest pain radiating to left arm"* or *"fever 3 days, child 2yo"*. These carry clinical signal that is lost when the intake system categorizes them into `chief_complaint_system`.

**Pipeline:**
1. **Clinical preprocessing** — lowercase, strip punctuation, expand abbreviations (SOB → shortness of breath, CP → chest pain, etc.)
2. **Keyword risk extraction** — binary flags for high-acuity clinical keywords (e.g. *diaphoresis*, *syncope*, *LOC*)
3. **TF-IDF + Truncated SVD (LSA)** — 50-component latent semantic embedding; handles synonymy and co-occurrence
4. **Late fusion** — NLP features concatenated to structured feature matrix

In [ ]:
# ── Clinical text abbreviation expansion ──────────────────────────────────────
ABBREV = {
    r'\bsob\b': 'shortness of breath',
    r'\bcp\b':  'chest pain',
    r'\bloc\b': 'loss of consciousness',
    r'\bams\b': 'altered mental status',
    r'\bn/v\b': 'nausea vomiting',
    r'\bnvd\b': 'nausea vomiting diarrhea',
    r'\bha\b':  'headache',
    r'\bhtn\b': 'hypertension',
    r'\bdm\b':  'diabetes',
    r'\bmi\b':  'myocardial infarction',
    r'\bcva\b': 'stroke',
    r'\bpe\b':  'pulmonary embolism',
    r'\bdvt\b': 'deep vein thrombosis',
    r'\buti\b': 'urinary tract infection',
    r'\burti\b':'upper respiratory infection',
}

HIGH_ACUITY_KW = [
    'diaphoresis', 'sweating', 'syncope', 'fainted', 'unconscious',
    'unresponsive', 'altered', 'confusion', 'seizure', 'stroke',
    'radiating', 'crushing', 'tearing', 'worst ever', 'severe',
    'cannot breathe', 'difficulty breathing', 'shortness of breath',
    'chest pain', 'palpitation', 'racing heart', 'hemoptysis',
    'hematuria', 'blood', 'bleeding', 'hemorrhage', 'trauma',
    'accident', 'overdose', 'poisoning', 'anaphylaxis', 'allergic',
    'paralysis', 'weakness', 'numbness', 'vision loss', 'speech',
    'face droop', 'arm weakness', 'pregnancy', 'labor', 'sepsis'
]

def preprocess_complaint(text):
    if pd.isna(text):
        return ''
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9\s/]', ' ', text)
    for pattern, replacement in ABBREV.items():
        text = re.sub(pattern, replacement, text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_keyword_features(texts):
    results = []
    for text in texts:
        row = {
            f'kw_{re.sub(r"[^a-z0-9]", "_", kw[:20]).strip("_")}': int(kw in text)
            for kw in HIGH_ACUITY_KW
        }
        row['kw_total'] = sum(row.values())
        results.append(row)
    return pd.DataFrame(results, index=texts.index)

train_text = train['complaint_text'].fillna('').apply(preprocess_complaint)
test_text  = test['complaint_text'].fillna('').apply(preprocess_complaint)

tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 3),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True
)
tfidf.fit(train_text)

svd = TruncatedSVD(n_components=60, random_state=SEED)
svd.fit(tfidf.transform(train_text))

def get_nlp_features(texts):
    tfidf_mat = tfidf.transform(texts)
    lsa = svd.transform(tfidf_mat)
    lsa_df = pd.DataFrame(
        lsa,
        columns=[f'lsa_{i}' for i in range(lsa.shape[1])],
        index=texts.index
    )
    kw_df = extract_keyword_features(texts)
    return pd.concat([lsa_df, kw_df], axis=1)

X_nlp_train = get_nlp_features(train_text)
X_nlp_test  = get_nlp_features(test_text)

print(f'NLP feature matrix (train): {X_nlp_train.shape}')
print(f'LSA variance explained: {svd.explained_variance_ratio_.sum():.1%}')
print(f'High-acuity keyword columns: {len([c for c in X_nlp_train.columns if c.startswith("kw_")])}')

In [ ]:
# ── Late fusion: concatenate structured + NLP features ────────────────────────
X_full       = pd.concat([X_struct.reset_index(drop=True),
                           X_nlp_train.reset_index(drop=True)], axis=1)
X_test_full  = pd.concat([X_test_struct.reset_index(drop=True),
                           X_nlp_test.reset_index(drop=True)], axis=1)

print(f'Full feature matrix: {X_full.shape}')
print(f'  Structured: {X_struct.shape[1]} | NLP: {X_nlp_train.shape[1]}')

## 6. Model Training — Calibrated Stacking Ensemble

### Architecture

```
                 ┌──────────────┐
  X_full  ──────►│   LightGBM   │──┐
                 └──────────────┘  │
                 ┌──────────────┐  │   ┌─────────────────────┐
  X_full  ──────►│   XGBoost    │──┼──►│  Meta-Learner (LR)  │──► P(ESI 1-5)
                 └──────────────┘  │   └─────────────────────┘
                 ┌──────────────┐  │
  X_struct──────►│  Ord. Ridge  │──┘
                 └──────────────┘
```

- **Ordinal Ridge** uses an ordinal encoding trick (cumulative binary decomposition) — captures that ESI 1 and ESI 3 disagreement is worse than ESI 2 and ESI 3
- **Stacking with 5-fold CV** — meta-features trained on OOF predictions to prevent leakage
- **Isotonic calibration** — applied per class after stacking, producing calibrated probabilities for downstream URS

### Evaluation targets

| Metric | Why |
|---|---|
| Quadratic weighted κ | Standard for ordinal agreement; comparable to human inter-rater |
| Macro AUC (OvR) | Discrimination across all 5 classes |
| ESI 1–2 sensitivity | Safety-critical: we care more about missing urgent cases |
| Mean Brier score | Probability calibration quality |

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_full, y, test_size=0.15, random_state=SEED, stratify=y
)

imputer = SimpleImputer(strategy='median')
X_tr_imp = pd.DataFrame(
    imputer.fit_transform(X_tr),
    columns=X_tr.columns,
    index=X_tr.index
)
X_val_imp = pd.DataFrame(
    imputer.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)
X_test_imp = pd.DataFrame(
    imputer.transform(X_test_full),
    columns=X_test_full.columns,
    index=X_test_full.index
)

scaler = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_tr_imp)
X_val_sc = scaler.transform(X_val_imp)

print(f'Train: {X_tr_imp.shape}  Val: {X_val_imp.shape}  Test: {X_test_imp.shape}')

In [ ]:
# ── Ordinal Logistic Regression (Cumulative Decomposition) ────────────────────
# Decompose 5-class ordinal into 4 binary problems: P(Y > k) for k=1..4
# Then reconstruct P(Y=k) from the cumulative probabilities

class OrdinalClassifier:
    """Proportional-odds approximation via stacked binary LR classifiers."""
    def __init__(self, C=1.0, seed=42):
        self.C = C
        self.seed = seed
        self.clfs = {}
        self.classes_ = None

    def fit(self, X, y):
        self.classes_ = np.sort(np.unique(y))
        for k in self.classes_[:-1]:
            y_bin = (y > k).astype(int)
            clf = LogisticRegression(C=self.C, max_iter=500,
                                     solver='lbfgs', random_state=self.seed)
            clf.fit(X, y_bin)
            self.clfs[k] = clf
        return self

    def predict_proba(self, X):
        cumulative = {}
        for k, clf in self.clfs.items():
            cumulative[k] = clf.predict_proba(X)[:, 1]  # P(Y > k)
        n = X.shape[0]
        K = len(self.classes_)
        proba = np.zeros((n, K))
        # P(Y=1) = 1 - P(Y>1)
        proba[:, 0] = 1 - cumulative[self.classes_[0]]
        for i, k in enumerate(self.classes_[1:-1]):
            proba[:, i+1] = cumulative[self.classes_[i]] - cumulative[k]
        # P(Y=K) = P(Y > K-1)
        proba[:, -1] = cumulative[self.classes_[-2]]
        proba = np.clip(proba, 0, 1)
        proba = proba / proba.sum(axis=1, keepdims=True)
        return proba

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

print('OrdinalClassifier defined.')

In [ ]:
# ── Base learners ─────────────────────────────────────────────────────────────
lgbm_base = lgb.LGBMClassifier(
    n_estimators=600,
    learning_rate=0.04,
    num_leaves=80,
    max_depth=8,
    min_child_samples=25,
    colsample_bytree=0.75,
    subsample=0.80,
    reg_alpha=0.05,
    reg_lambda=0.15,
    random_state=SEED,
    verbose=-1,
    class_weight='balanced'
)

xgb_base = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    n_estimators=500,
    learning_rate=0.04,
    max_depth=7,
    colsample_bytree=0.75,
    subsample=0.80,
    gamma=0.05,
    reg_alpha=0.05,
    reg_lambda=1.0,
    random_state=SEED,
    eval_metric='mlogloss',
    verbosity=0
)

ordinal_lr = OrdinalClassifier(C=1.0, seed=SEED)

print('Training LightGBM...')
lgbm_base.fit(X_tr_imp, y_tr)
print('  LightGBM ✓')

print('Training XGBoost...')
xgb_base.fit(X_tr_imp, y_tr - 1)
print('  XGBoost ✓')

print('Training Ordinal LR...')
ordinal_lr.fit(X_tr_sc, y_tr)
print('  Ordinal LR ✓')

In [ ]:
# ── Stacking: build OOF meta-features on TRAIN SPLIT ONLY ────────────────────
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
n_classes = 5

X_tr_reset = X_tr.reset_index(drop=True)
y_tr_reset = y_tr.reset_index(drop=True)

oof_lgbm  = np.zeros((len(y_tr_reset), n_classes))
oof_xgb   = np.zeros((len(y_tr_reset), n_classes))
oof_ordlr = np.zeros((len(y_tr_reset), n_classes))

print(f'Building OOF meta-features on training split ({N_FOLDS} folds)...')
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tr_reset, y_tr_reset)):
    Xf_tr_raw  = X_tr_reset.iloc[tr_idx]
    Xf_val_raw = X_tr_reset.iloc[val_idx]
    yf_tr      = y_tr_reset.iloc[tr_idx]
    yf_val     = y_tr_reset.iloc[val_idx]

    fold_imputer = SimpleImputer(strategy='median')
    Xf_tr = pd.DataFrame(
        fold_imputer.fit_transform(Xf_tr_raw),
        columns=Xf_tr_raw.columns,
        index=Xf_tr_raw.index
    )
    Xf_val = pd.DataFrame(
        fold_imputer.transform(Xf_val_raw),
        columns=Xf_val_raw.columns,
        index=Xf_val_raw.index
    )

    fold_scaler = StandardScaler()
    Xf_tr_sc = fold_scaler.fit_transform(Xf_tr)
    Xf_val_sc = fold_scaler.transform(Xf_val)

    m_lgb = lgb.LGBMClassifier(
        n_estimators=600,
        learning_rate=0.04,
        num_leaves=80,
        max_depth=8,
        min_child_samples=25,
        colsample_bytree=0.75,
        subsample=0.80,
        reg_alpha=0.05,
        reg_lambda=0.15,
        random_state=SEED + fold,
        verbose=-1,
        class_weight='balanced'
    )
    m_lgb.fit(Xf_tr, yf_tr)
    oof_lgbm[val_idx] = m_lgb.predict_proba(Xf_val)

    m_xgb = xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=n_classes,
        n_estimators=500,
        learning_rate=0.04,
        max_depth=7,
        colsample_bytree=0.75,
        subsample=0.80,
        gamma=0.05,
        reg_alpha=0.05,
        reg_lambda=1.0,
        random_state=SEED + fold,
        eval_metric='mlogloss',
        verbosity=0
    )
    m_xgb.fit(Xf_tr, yf_tr - 1)
    oof_xgb[val_idx] = m_xgb.predict_proba(Xf_val)

    m_ord = OrdinalClassifier(C=1.0, seed=SEED + fold)
    m_ord.fit(Xf_tr_sc, yf_tr)
    oof_ordlr[val_idx] = m_ord.predict_proba(Xf_val_sc)

    kf = cohen_kappa_score(
        yf_val,
        np.argmax(oof_lgbm[val_idx], axis=1) + 1,
        weights='quadratic'
    )
    print(f'  Fold {fold + 1} LGB OOF κ: {kf:.4f}')

print('OOF meta-features complete.')

In [ ]:
# ── Train meta-learner on OOF predictions from training split ────────────────
meta_X_train = np.concatenate([oof_lgbm, oof_xgb, oof_ordlr], axis=1)
meta_scaler = StandardScaler()
meta_X_train_sc = meta_scaler.fit_transform(meta_X_train)

meta_lr = LogisticRegression(C=5.0, max_iter=1000, random_state=SEED)
meta_lr.fit(meta_X_train_sc, y_tr_reset)
print('Meta-learner (LR) trained on train-split OOF stack.')

meta_lr_cal = CalibratedClassifierCV(meta_lr, method='isotonic', cv=5)
meta_lr_cal.fit(meta_X_train_sc, y_tr_reset)
print('Calibration applied (isotonic, 5-fold) on train-split meta features.')

def predict_ensemble(X_imp, X_sc):
    p_lgb = lgbm_base.predict_proba(X_imp)
    p_xgb = xgb_base.predict_proba(X_imp)
    p_ord = ordinal_lr.predict_proba(X_sc)
    stack = np.concatenate([p_lgb, p_xgb, p_ord], axis=1)
    stack_sc = meta_scaler.transform(stack)
    return meta_lr_cal.predict_proba(stack_sc)

proba_val = predict_ensemble(X_val_imp, X_val_sc)
y_pred_val = np.argmax(proba_val, axis=1) + 1

kappa   = cohen_kappa_score(y_val, y_pred_val, weights='quadratic')
acc     = accuracy_score(y_val, y_pred_val)
y_val_bin = label_binarize(y_val, classes=[1,2,3,4,5])
auc_mac = roc_auc_score(y_val_bin, proba_val, multi_class='ovr', average='macro')
brier   = np.mean([brier_score_loss(y_val_bin[:, i], proba_val[:, i]) for i in range(5)])

esi12_true = (y_val <= 2).astype(int)
esi12_pred = (y_pred_val <= 2).astype(int)
esi12_sens = (esi12_true & esi12_pred).sum() / (esi12_true.sum() + 1e-9)

print('=' * 58)
print('          STACKED ENSEMBLE — VALIDATION RESULTS')
print('=' * 58)
print(f'  Quadratic Weighted κ : {kappa:.4f}  (human range: 0.30-0.50)')
print(f'  Overall Accuracy      : {acc:.4f}')
print(f'  Macro AUC (OvR)       : {auc_mac:.4f}')
print(f'  Mean Brier Score      : {brier:.4f}')
print(f'  ESI 1-2 Sensitivity   : {esi12_sens:.4f}  ← SAFETY CRITICAL')
print('=' * 58)
print()
print(classification_report(
    y_val,
    y_pred_val,
    target_names=['ESI 1','ESI 2','ESI 3','ESI 4','ESI 5']
))

In [ ]:
# ── Performance visualizations ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Stacked Ensemble — Model Performance', fontsize=14, fontweight='bold')

# Confusion matrix
ax = axes[0]
cm = confusion_matrix(y_val, y_pred_val)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax, linewidths=0.5,
            xticklabels=['ESI 1','ESI 2','ESI 3','ESI 4','ESI 5'],
            yticklabels=['ESI 1','ESI 2','ESI 3','ESI 4','ESI 5'])
ax.set_title(f'Normalized Confusion Matrix\nκ = {kappa:.3f}', fontweight='bold')
ax.set_ylabel('True ESI')
ax.set_xlabel('Predicted ESI')

# Calibration curves
ax = axes[1]
for i, (esi, col) in enumerate(zip([1,2,3,4,5], ESI_COLORS)):
    try:
        pt, pp = calibration_curve((y_val==esi).astype(int), proba_val[:,i], n_bins=8)
        ax.plot(pp, pt, 'o-', color=col, label=f'ESI {esi}', alpha=0.85)
    except Exception:
        pass
ax.plot([0,1],[0,1],'k--', label='Perfect')
ax.set_title('Probability Calibration\n(isotonic regression)', fontweight='bold')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction Positives')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ESI-level AUC bar chart
ax = axes[2]
esi_aucs = []
for i in range(5):
    try:
        a = roc_auc_score(y_val_bin[:,i], proba_val[:,i])
    except Exception:
        a = 0
    esi_aucs.append(a)
bars = ax.bar([f'ESI {i+1}' for i in range(5)], esi_aucs, color=ESI_COLORS, edgecolor='white')
ax.axhline(0.5, ls='--', color='gray', alpha=0.6, label='Random (0.5)')
ax.axhline(np.mean(esi_aucs), ls='--', color='navy', alpha=0.6, label=f'Macro AUC ({np.mean(esi_aucs):.3f})')
for bar, auc_val in zip(bars, esi_aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{auc_val:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylim(0.4, 1.05)
ax.set_title('AUC by ESI Level (OvR)', fontweight='bold')
ax.set_ylabel('AUC')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('model_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Importance & Clinical Interpretability

Interpretability is not optional in clinical AI. We report:
1. **LightGBM feature importance** (gain-based) — which features drive splits
2. **NLP keyword contribution** — which free-text terms most strongly predict high acuity
3. **Partial dependence** for the three most important continuous vitals

In [ ]:
# Structured feature importance
all_feature_importance = pd.DataFrame({
    'feature': X_tr_imp.columns,
    'importance': lgbm_base.feature_importances_
})

struct_importance = (
    all_feature_importance[all_feature_importance['feature'].isin(X_struct.columns)]
    .sort_values('importance', ascending=False)
    .head(30)
)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('Feature Importance & Clinical Interpretability', fontsize=14, fontweight='bold')

ax = axes[0]
def color_by_domain(feat):
    if any(v in feat for v in ['hr_','sbp','spo2','resp','temp','heart','hypo','hyper',
                                'tachy','brady','shock','news2','pulse','gcs','pain']):
        return '#c62828'
    if any(v in feat for v in ['kw_','lsa_','complaint']):
        return '#1565c0'
    if any(v in feat for v in ['hx_','comorbid']):
        return '#2e7d32'
    return '#6a1b9a'

colors = [color_by_domain(f) for f in struct_importance['feature']]
struct_importance.plot(
    kind='barh', x='feature', y='importance',
    ax=ax, color=colors, edgecolor='white', linewidth=0.5, legend=False
)
ax.set_title('Top 30 Structured Features — LightGBM Gain', fontweight='bold')
ax.set_xlabel('Gain Importance')
patches = [
    mpatches.Patch(color='#c62828', label='Vitals/Clinical'),
    mpatches.Patch(color='#1565c0', label='NLP/Text'),
    mpatches.Patch(color='#2e7d32', label='Comorbidities'),
    mpatches.Patch(color='#6a1b9a', label='Demographics/Context'),
]
ax.legend(handles=patches, fontsize=9)

ax = axes[1]
kw_cols = [c for c in X_nlp_train.columns if c.startswith('kw_') and c != 'kw_total']
if kw_cols:
    kw_mat = X_nlp_train.loc[y.index, kw_cols].copy()
    kw_corr = kw_mat.corrwith((y <= 2).astype(float)).sort_values(ascending=False).head(20)
    kw_corr_colors = ['#c62828' if v > 0 else '#1565c0' for v in kw_corr.values]
    kw_corr.plot(kind='barh', ax=ax, color=kw_corr_colors, edgecolor='white', linewidth=0.5)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title('NLP Keyword Correlation with High Acuity (ESI 1-2)', fontweight='bold')
    ax.set_xlabel('Point-Biserial Correlation with ESI ≤ 2')
else:
    ax.text(0.5, 0.5, 'No keyword features found', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Fairness Analysis — Systematic Undertriage Bias

**Scientific framing:** We treat triage acuity as a *noisy measurement* of true patient severity. If that noise is systematically higher for certain demographic groups, the result is *undertriage* — assigning a lower acuity score (less urgent) than the patient's clinical status warrants.

We operationalize undertriage as: **model predicts ESI ≤ nurse-assigned ESI − 1** (model judges the patient as more urgent than the nurse did). This is a model-relative proxy; in real data one would use acuity proxies like ICU admission or 72-hour return visit.

We then test whether the *probability* of model–nurse disagreement (in the direction of undertriage) varies across demographic subgroups using:
- Relative risk ratios with 95% bootstrap CI
- Fisher's exact test (or chi-squared for larger cells)
- Multivariable logistic regression controlling for complaint severity

In [ ]:
# ── Compute undertriage proxy on validation set ───────────────────────────────
val_df = train.iloc[y_val.index].copy()
val_df['y_true']        = y_val.values
val_df['y_pred']        = y_pred_val
val_df['p_esi12']       = proba_val[:, 0] + proba_val[:, 1]

# Model-relative undertriage: model says patient is more urgent than nurse assigned
val_df['model_undertriage'] = (val_df['y_pred'] < val_df['y_true']).astype(int)

overall_ut_rate = val_df['model_undertriage'].mean()
print(f'Overall model-flagged undertriage rate: {overall_ut_rate:.2%}')

def rr_analysis(df, group_col):
    """Relative risk of model-flagged undertriage by group, with 95% bootstrap CI."""
    if group_col not in df.columns:
        return pd.DataFrame()
    overall = df['model_undertriage'].mean()
    rows = []
    for grp in df[group_col].dropna().unique():
        sub = df[df[group_col] == grp]
        if len(sub) < 20:
            continue
        ut_rate = sub['model_undertriage'].mean()
        rr = ut_rate / (overall + 1e-9)
        # Bootstrap CI for RR
        boot_rrs = []
        for _ in range(500):
            s = sub.sample(len(sub), replace=True)
            boot_rrs.append(s['model_undertriage'].mean() / (overall + 1e-9))
        ci_lo, ci_hi = np.percentile(boot_rrs, [2.5, 97.5])
        # Fisher's exact test
        ct = pd.crosstab(df[group_col] == grp, df['model_undertriage'])
        if ct.shape == (2, 2):
            _, p_val = stats.fisher_exact(ct)
        else:
            p_val = 1.0
        rows.append({
            'group': grp, 'n': len(sub), 'ut_rate': ut_rate,
            'rr': rr, 'ci_lo': ci_lo, 'ci_hi': ci_hi, 'p_value': p_val
        })
    return pd.DataFrame(rows).sort_values('rr', ascending=False)

# Analyze by available demographic columns
for col in ['sex', 'insurance_type', 'language', 'age_group']:
    rr_df = rr_analysis(val_df, col)
    if not rr_df.empty:
        print(f'\n=== Undertriage RR by {col} ===')
        print(rr_df[['group','n','ut_rate','rr','ci_lo','ci_hi','p_value']].to_string(index=False))

In [ ]:
# ── Multivariable logistic regression controlling for clinical severity ────────
# Controls: NEWS2 proxy, chief complaint system, age
demo_cols = [c for c in ['sex', 'insurance_type', 'language', 'age_group', 'arrival_mode'] if c in val_df.columns]
clinical_controls = [c for c in ['news2_score', 'news2_proxy', 'shock_index',
                                   'chief_complaint_system', 'age', 'num_comorbidities'] if c in val_df.columns]

bias_df = val_df[demo_cols + clinical_controls + ['model_undertriage']].copy()
bias_df = pd.get_dummies(bias_df, columns=[c for c in demo_cols if c in bias_df.columns
                                            and bias_df[c].dtype == object],
                          drop_first=True)
bias_df = pd.get_dummies(bias_df, columns=[c for c in clinical_controls if c in bias_df.columns
                                            and bias_df[c].dtype == object],
                          drop_first=True)

X_bias = bias_df.drop('model_undertriage', axis=1).fillna(0)
y_bias = bias_df['model_undertriage']

X_bias_sc = StandardScaler().fit_transform(X_bias)
lr_adj = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=SEED)
lr_adj.fit(X_bias_sc, y_bias)

coefs = pd.Series(lr_adj.coef_[0], index=X_bias.columns)
OR = np.exp(coefs).sort_values(ascending=False)

print('Adjusted Odds Ratios for Model-Flagged Undertriage:')
print('(Controlling for clinical severity indicators)')
display_or = OR[(OR > 1.1) | (OR < 0.9)].head(20)
print(display_or.to_string())

In [ ]:
# ── Bias visualization ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(17, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('Systematic Undertriage Bias Analysis', fontsize=14, fontweight='bold')

# Panel A: Undertriage rate by insurance type (proxy for socioeconomic status)
ax1 = fig.add_subplot(gs[0, 0])
if 'insurance_type' in val_df.columns:
    ins_ut = val_df.groupby('insurance_type')['model_undertriage'].agg(['mean','count'])
    ins_ut = ins_ut[ins_ut['count'] > 30].sort_values('mean')
    colors_a = ['#c62828' if v > overall_ut_rate*1.1 else '#2e7d32' for v in ins_ut['mean']]
    ax1.barh(ins_ut.index, ins_ut['mean'], color=colors_a, alpha=0.85)
    ax1.axvline(overall_ut_rate, ls='--', color='navy', lw=2, label=f'Overall ({overall_ut_rate:.2f})')
    ax1.set_title('A. Undertriage Rate\nby Insurance Type', fontweight='bold')
    ax1.set_xlabel('Model-Flagged Undertriage Rate')
    ax1.legend(fontsize=8)

# Panel B: Undertriage by sex x complaint system
ax2 = fig.add_subplot(gs[0, 1])
sex_col = 'sex' if 'sex' in val_df.columns else None
cc_col_bias = 'chief_complaint_system' if 'chief_complaint_system' in val_df.columns else None
if sex_col and cc_col_bias:
    top_complaints = val_df[cc_col_bias].value_counts().head(6).index
    sub = val_df[val_df[cc_col_bias].isin(top_complaints)]
    pivot = sub.pivot_table(values='model_undertriage', index=cc_col_bias,
                             columns=sex_col, aggfunc='mean')
    pivot.plot(kind='bar', ax=ax2, color=['#1565c0','#c62828'][:len(pivot.columns)],
               edgecolor='white', alpha=0.85)
    ax2.set_title('B. Undertriage Rate by\nSex × Chief Complaint', fontweight='bold')
    ax2.set_ylabel('Undertriage Rate')
    ax2.tick_params(axis='x', rotation=45)
    ax2.legend(title='Sex', fontsize=8)

# Panel C: Age-gradient undertriage
ax3 = fig.add_subplot(gs[0, 2])
if 'age' in val_df.columns:
    val_df['age_bin'] = pd.cut(val_df['age'], bins=[0,18,45,65,75,100],
                                labels=['<18','18-44','45-64','65-74','75+'])
    age_ut = val_df.groupby('age_bin')['model_undertriage'].mean()
    age_n  = val_df.groupby('age_bin')['model_undertriage'].count()
    colors_c = ['#c62828' if v > overall_ut_rate*1.05 else '#2e7d32' for v in age_ut]
    ax3.bar(age_ut.index.astype(str), age_ut.values, color=colors_c, alpha=0.85, edgecolor='white')
    ax3.axhline(overall_ut_rate, ls='--', color='navy', lw=2, label=f'Overall ({overall_ut_rate:.2f})')
    for i, (v, n) in enumerate(zip(age_ut.values, age_n.values)):
        ax3.text(i, v + 0.002, f'n={n}', ha='center', fontsize=8)
    ax3.set_title('C. Undertriage Rate\nby Age Group', fontweight='bold')
    ax3.set_ylabel('Undertriage Rate')
    ax3.legend(fontsize=8)

# Panel D: Adjusted OR forest plot
ax4 = fig.add_subplot(gs[1, :2])
demo_OR = OR[(OR > 1.05) | (OR < 0.95)].head(15).sort_values()
y_pos = range(len(demo_OR))
colors_d = ['#c62828' if v > 1 else '#2e7d32' for v in demo_OR.values]
ax4.barh(y_pos, demo_OR.values - 1, left=1, color=colors_d, alpha=0.8, edgecolor='white')
ax4.axvline(1.0, color='black', lw=1.5, label='OR=1 (no effect)')
ax4.set_yticks(list(y_pos))
ax4.set_yticklabels(demo_OR.index, fontsize=9)
ax4.set_xlabel('Adjusted Odds Ratio for Model-Flagged Undertriage')
ax4.set_title('D. Multivariable Adjusted ORs for Undertriage\n(controlling for clinical severity)',
              fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(axis='x', alpha=0.3)

# Panel E: P(high acuity) distribution: undertriaged vs correct
ax5 = fig.add_subplot(gs[1, 2])
ax5.hist(val_df[val_df['model_undertriage']==1]['p_esi12'], bins=30,
         alpha=0.7, color='#c62828', label='Model-flagged undertriaged', density=True)
ax5.hist(val_df[val_df['model_undertriage']==0]['p_esi12'], bins=30,
         alpha=0.7, color='#2e7d32', label='Correct triage', density=True)
ax5.set_title('E. P(ESI 1-2): Undertriaged\nvs Correct Patients', fontweight='bold')
ax5.set_xlabel('P(high acuity | model)')
ax5.set_ylabel('Density')
ax5.legend(fontsize=8)

plt.savefig('bias_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Post-Hoc Fairness Correction — Equalized Odds Threshold Adjustment

**Motivation:** Even a highly accurate model trained on biased nurse labels will *encode* those biases. Simply training harder won't fix this.

**Method:** We apply a post-hoc equalized odds correction (Hardt et al., 2016) to high-risk subgroups. For each protected group $g$, we find a group-specific threshold $\tau_g$ such that the model's *recall on ESI 1–2 patients* is equalized across groups.

Concretely: rather than using $\hat{y} = \arg\max P(\text{ESI}|x)$ for all patients, for patients in undertriage-risk groups we **lower the threshold** for flagging ESI 1–2, accepting a slightly higher false positive rate in exchange for not missing high-acuity patients.

This is clinically defensible: the cost of undertriage (missed high-acuity patient) is much higher than overtriage (a patient given priority who turns out to be lower acuity).

In [ ]:
def compute_group_threshold(df, proba_col='p_esi12', label_col='y_true',
                             target_recall=0.85, esi_threshold=2):
    """
    Find the P(ESI<=2) threshold that achieves at least `target_recall`
    on true ESI 1-2 patients.
    """
    high_acuity = (df[label_col] <= esi_threshold).astype(int)
    if high_acuity.sum() < 10:
        return 0.5  # fallback
    best_thresh = 0.5
    for t in np.linspace(0.05, 0.80, 150):
        preds = (df[proba_col] >= t).astype(int)
        recall = (high_acuity & preds).sum() / (high_acuity.sum() + 1e-9)
        if recall >= target_recall:
            best_thresh = t
            break
    return best_thresh


# Define undertriage-risk subgroups based on literature and bias analysis above
# In production, this would be updated from prospective monitoring
RISK_GROUP_COLS = {}
if 'insurance_type' in val_df.columns:
    # Example: Medicaid/uninsured patients often undertriaged (SES proxy)
    RISK_GROUP_COLS['insurance_risk'] = val_df['insurance_type'].isin(['Medicaid','Uninsured','Self-pay'])
if 'age' in val_df.columns:
    RISK_GROUP_COLS['elderly'] = val_df['age'] >= 75
if 'language' in val_df.columns:
    non_english = val_df['language'].astype(str).str.lower().apply(
        lambda x: 'english' not in x and x not in ['nan', '']
    )
    RISK_GROUP_COLS['non_english'] = non_english

# Compute group-specific thresholds
global_threshold = compute_group_threshold(val_df, target_recall=0.82)
print(f'Global P(ESI≤2) threshold for 82% recall: {global_threshold:.3f}')

group_thresholds = {}
for grp_name, mask in RISK_GROUP_COLS.items():
    if mask.sum() < 20:
        continue
    sub = val_df[mask]
    t = compute_group_threshold(sub, target_recall=0.88)  # stricter for risk groups
    group_thresholds[grp_name] = t
    print(f'  {grp_name}: n={mask.sum()}, threshold={t:.3f}')


def fairness_corrected_predict(df_rows, probas):
    """
    Apply group-specific thresholds to flag high-acuity cases.
    Returns: array of 1 (high acuity flagged) or 0
    """
    p_high = probas[:, 0] + probas[:, 1]
    threshold = np.full(len(df_rows), global_threshold, dtype=float)

    for grp_name, mask in RISK_GROUP_COLS.items():
        if grp_name in group_thresholds:
            grp_mask = mask.values if hasattr(mask, 'values') else np.array(mask)
            threshold[grp_mask] = np.minimum(
                threshold[grp_mask],
                group_thresholds[grp_name]
            )

    return (p_high >= threshold).astype(int)


# Compare recall before/after correction
high_true_val = (val_df['y_true'] <= 2).astype(int)
high_pred_base = (val_df['p_esi12'] >= 0.5).astype(int)
high_pred_corrected = fairness_corrected_predict(
    val_df.reset_index(drop=True), proba_val
)

print('\n=== Fairness Correction Impact ===')
print(f'Recall (ESI 1-2), Baseline:   {(high_true_val & high_pred_base).sum() / (high_true_val.sum()+1e-9):.3f}')
print(f'Recall (ESI 1-2), Corrected:  {(high_true_val.values & high_pred_corrected).sum() / (high_true_val.sum()+1e-9):.3f}')
print(f'Precision, Baseline:   {(high_true_val & high_pred_base).sum() / (high_pred_base.sum()+1e-9):.3f}')
print(f'Precision, Corrected:  {(high_true_val.values & high_pred_corrected).sum() / (high_pred_corrected.sum()+1e-9):.3f}')

## 10. Undertriage Risk Score (URS) — Deployable Clinical Output

Rather than outputting raw ESI predictions, we produce a single **Undertriage Risk Score (URS)** per patient — a calibrated probability that the patient has been (or would be) undertriaged. This is the actionable output for a clinical decision support interface.

$$\text{URS} = w_1 \cdot P(\text{ESI} \leq 2 | x) + w_2 \cdot \mathbb{1}[\text{model disagrees with nurse}] + w_3 \cdot \text{NEWS2-proxy} / 7$$

URS is normalized to [0, 1] and calibrated against validation undertriage labels.

In [ ]:
def compute_urs(proba, nurse_esi, news2_scores, max_news2=7):
    """
    Undertriage Risk Score — normalized, interpretable composite.
    """
    p_high = proba[:, 0] + proba[:, 1]             # P(ESI 1-2)
    model_esi = np.argmax(proba, axis=1) + 1
    model_disagrees = (model_esi < nurse_esi).astype(float)
    news2_norm = np.clip(np.nan_to_num(news2_scores, nan=0) / max_news2, 0, 1)

    urs = 0.50 * p_high + 0.30 * model_disagrees + 0.20 * news2_norm
    urs = np.clip(urs, 0, 1)
    return urs


news2_val = val_df.get('news2_score', val_df.get('news2_proxy', pd.Series(np.zeros(len(val_df)), index=val_df.index))).values

urs_val = compute_urs(proba_val, val_df['y_true'].values, news2_val)
val_df['urs'] = urs_val

# Visualize URS distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Undertriage Risk Score (URS) — Validation Set', fontsize=13, fontweight='bold')

ax = axes[0]
for esi, col in zip([1,2,3,4,5], ESI_COLORS):
    sub_urs = val_df[val_df['y_true'] == esi]['urs']
    ax.hist(sub_urs, bins=30, alpha=0.55, color=col, label=f'ESI {esi}', density=True)
ax.set_title('URS Distribution by True ESI\n(higher URS → higher concern)', fontweight='bold')
ax.set_xlabel('Undertriage Risk Score')
ax.legend(fontsize=8)

ax = axes[1]
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
labels = ['Very Low\n(0–0.2)', 'Low\n(0.2–0.4)', 'Moderate\n(0.4–0.6)',
          'High\n(0.6–0.8)', 'Very High\n(0.8–1.0)']
val_df['urs_cat'] = pd.cut(val_df['urs'], bins=bins, labels=labels)
urs_acuity = val_df.groupby('urs_cat')['y_true'].mean()
bar_c = ['#2e7d32','#66bb6a','#f9a825','#ef6c00','#c62828']
ax.bar(urs_acuity.index.astype(str), urs_acuity.values, color=bar_c, edgecolor='white')
ax.set_title('Mean True ESI by URS Category\n(lower ESI = more urgent)', fontweight='bold')
ax.set_ylabel('Mean True ESI Level')
ax.set_xlabel('URS Category')
for i, v in enumerate(urs_acuity.values):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=10, fontweight='bold')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('urs_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('URS summary statistics by true ESI:')
print(val_df.groupby('y_true')['urs'].describe().round(3).to_string())

## 11. Clinical Decision Support — Alert System Prototype

The alert system is the clinical-facing interface. For each patient, it produces a structured output that a triage nurse or charge nurse can act on without needing to understand the underlying ML.

**Alert tiers:**

| Tier | Trigger | Action |
|---|---|---|
| 🔴 CRITICAL | URS ≥ 0.70 OR (model ESI ≤ nurse ESI − 2) | Immediate secondary review by physician |
| 🟡 WARNING | URS ≥ 0.45 OR (model ESI ≤ nurse ESI − 1) | Nurse reassessment within 15 min |
| 🟢 OK | URS < 0.45 AND model agrees | No additional action |

In [ ]:
def triage_alert_system(patient_row: dict, assigned_esi: int,
                         model_probas: np.ndarray,
                         news2_score: float = 0,
                         urs_critical_threshold: float = 0.70,
                         urs_warning_threshold: float = 0.45) -> dict:
    """
    Fairness-aware, URS-based clinical alert system.

    Parameters
    ----------
    patient_row : dict  — patient demographics and presenting features
    assigned_esi : int  — triage nurse's ESI score (1–5)
    model_probas : np.ndarray — calibrated P(ESI=1..5) from ensemble
    news2_score : float — NEWS2 or proxy score (default 0 if unavailable)
    urs_critical_threshold : float — URS threshold for CRITICAL alert
    urs_warning_threshold  : float — URS threshold for WARNING alert

    Returns
    -------
    dict with keys: alert, level, urs, model_esi, p_high_acuity,
                    reasons, recommended_action, demographic_flags
    """
    model_esi    = int(np.argmax(model_probas) + 1)
    p_high       = float(model_probas[0] + model_probas[1])
    urs          = float(compute_urs(
        model_probas.reshape(1, -1),
        np.array([assigned_esi]),
        np.array([news2_score])
    )[0])

    level   = 'OK'
    reasons = []
    demo_flags = []

    # ── Vital sign CRITICAL triggers ─────────────────────────────────────────
    if patient_row.get('systolic_bp', 120) < 90:
        reasons.append('Hypotension (SBP < 90 mmHg)')
        level = 'CRITICAL'
    if patient_row.get('spo2', 98) < 90:
        reasons.append('Severe hypoxia (SpO₂ < 90%)')
        level = 'CRITICAL'
    if patient_row.get('gcs_total', 15) <= 8:
        reasons.append('GCS ≤ 8 (critical neurological status)')
        level = 'CRITICAL'

    # ── URS-based thresholds ──────────────────────────────────────────────────
    if urs >= urs_critical_threshold:
        level = 'CRITICAL'
        reasons.append(f'Undertriage Risk Score = {urs:.2f} (threshold: {urs_critical_threshold})')
    elif urs >= urs_warning_threshold:
        if level == 'OK':
            level = 'WARNING'
        reasons.append(f'Undertriage Risk Score = {urs:.2f} (threshold: {urs_warning_threshold})')

    # ── Model-nurse disagreement ──────────────────────────────────────────────
    if model_esi <= assigned_esi - 2:
        level = 'CRITICAL'
        reasons.append(f'Model predicts ESI {model_esi} vs nurse ESI {assigned_esi} (gap ≥ 2 levels)')
    elif model_esi <= assigned_esi - 1:
        if level == 'OK':
            level = 'WARNING'
        reasons.append(f'Model predicts ESI {model_esi} vs nurse ESI {assigned_esi} (gap 1 level)')

    # ── Demographic risk flags (transparent, non-deterministic) ───────────────
    age = patient_row.get('age', 50)
    insurance = str(patient_row.get('insurance_type', '')).lower()
    language  = str(patient_row.get('language', 'english')).lower()
    sex       = str(patient_row.get('sex', '')).lower()
    complaint = str(patient_row.get('chief_complaint_system', '')).lower()

    if age >= 75:
        demo_flags.append('Age ≥ 75 (elevated undertriage risk per literature)')
    if 'medicaid' in insurance or 'uninsured' in insurance or 'self' in insurance:
        demo_flags.append('Medicaid/uninsured (documented undertriage risk group)')
    if 'english' not in language and language not in ['nan', '', 'none']:
        demo_flags.append('Non-English primary language (language-barrier risk factor)')
    if sex == 'female' and 'chest' in complaint:
        demo_flags.append('Female + chest pain (Rathore effect — documented sex-differential)')

    # Apply lower threshold if demo flags present
    if demo_flags and urs >= urs_warning_threshold * 0.75 and level == 'OK':
        level = 'WARNING'
        reasons.append('Demographic risk factor(s) + elevated URS')

    alert = level != 'OK'

    recommended_action = {
        'CRITICAL': 'Immediate physician secondary assessment required',
        'WARNING' : 'Nurse reassessment within 15 minutes',
        'OK'      : 'No immediate action required'
    }[level]

    return {
        'alert'              : alert,
        'level'              : level,
        'urs'                : round(urs, 3),
        'model_esi'          : model_esi,
        'p_high_acuity'      : round(p_high, 3),
        'reasons'            : reasons,
        'demographic_flags'  : demo_flags,
        'recommended_action' : recommended_action,
    }


# ── Run alert system on validation set ───────────────────────────────────────
alert_results = []
val_reset = val_df.reset_index(drop=True)

for i in range(len(val_reset)):
    row = val_reset.iloc[i].to_dict()
    news2 = row.get('news2_score', row.get('news2_proxy', 0))
    result = triage_alert_system(
        patient_row=row,
        assigned_esi=int(row['y_true']),
        model_probas=proba_val[i],
        news2_score=float(news2) if not pd.isna(news2) else 0
    )
    result['true_esi']     = int(row['y_true'])
    result['assigned_esi'] = int(row['y_true'])
    alert_results.append(result)

alerts_df = pd.DataFrame(alert_results)

print('=== Alert System Performance — Validation Set ===')
print(f'Total patients     : {len(alerts_df):,}')
print(f'Alerts triggered   : {alerts_df["alert"].sum():,} ({alerts_df["alert"].mean():.1%})')
print(f'  CRITICAL         : {(alerts_df["level"]=="CRITICAL").sum():,}')
print(f'  WARNING          : {(alerts_df["level"]=="WARNING").sum():,}')

In [ ]:
# ── Demo: simulate a patient intake and show the full alert output ─────────────
DEMO_PATIENT = {
    'age'                  : 78,
    'sex'                  : 'Female',
    'insurance_type'       : 'Medicaid',
    'language'             : 'Spanish',
    'chief_complaint_system': 'Chest pain',
    'heart_rate'           : 108,
    'systolic_bp'          : 96,
    'spo2'                 : 93,
    'temperature_c'        : 37.2,
    'respiratory_rate'     : 22,
    'gcs_total'            : 14,
    'pain_score'           : 8,
}

# Simulate model output for demo (use median validation proba for ESI-2 patient)
demo_probas = np.array([0.08, 0.35, 0.32, 0.17, 0.08])
demo_alert  = triage_alert_system(
    patient_row=DEMO_PATIENT,
    assigned_esi=3,   # nurse assigned ESI 3
    model_probas=demo_probas,
    news2_score=4
)

print('=' * 60)
print('       TRIAGEGEIST CLINICAL DECISION SUPPORT OUTPUT')
print('=' * 60)
print(f'  Patient        : 78F, Spanish-speaking, Medicaid')
print(f'  Chief complaint: Chest pain')
print(f'  Nurse ESI      : ESI 3 (Urgent)')
print(f'  Model ESI      : ESI {demo_alert["model_esi"]} | P(ESI 1-2): {demo_alert["p_high_acuity"]:.1%}')
print(f'  URS            : {demo_alert["urs"]:.3f}')
print(f'  ─────────────────────────────────────────────────')
print(f'  ALERT LEVEL    : ⚠️  {demo_alert["level"]}')
print(f'  Action         : {demo_alert["recommended_action"]}')
print(f'  ─────────────────────────────────────────────────')
print(f'  Clinical reasons:')
for r in demo_alert['reasons']:
    print(f'    • {r}')
print(f'  Demographic flags (for transparency, not deterministic):')
for f in demo_alert['demographic_flags']:
    print(f'    ⚑ {f}')
print('=' * 60)

## 12. Test Set Predictions — Competition Submission

In [ ]:
# ── Refit ensemble on FULL TRAINING DATA and generate test predictions ───────
imputer_full = SimpleImputer(strategy='median')
X_full_imp = pd.DataFrame(
    imputer_full.fit_transform(X_full),
    columns=X_full.columns,
    index=X_full.index
)
X_test_imp = pd.DataFrame(
    imputer_full.transform(X_test_full),
    columns=X_test_full.columns,
    index=X_test_full.index
)

scaler_full = StandardScaler()
X_full_sc = scaler_full.fit_transform(X_full_imp)
X_test_sc = scaler_full.transform(X_test_imp)

lgbm_full = lgb.LGBMClassifier(
    n_estimators=600,
    learning_rate=0.04,
    num_leaves=80,
    max_depth=8,
    min_child_samples=25,
    colsample_bytree=0.75,
    subsample=0.80,
    reg_alpha=0.05,
    reg_lambda=0.15,
    random_state=SEED,
    verbose=-1,
    class_weight='balanced'
)
lgbm_full.fit(X_full_imp, y)

xgb_full = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    n_estimators=500,
    learning_rate=0.04,
    max_depth=7,
    colsample_bytree=0.75,
    subsample=0.80,
    gamma=0.05,
    reg_alpha=0.05,
    reg_lambda=1.0,
    random_state=SEED,
    eval_metric='mlogloss',
    verbosity=0
)
xgb_full.fit(X_full_imp, y - 1)

ordinal_full = OrdinalClassifier(C=1.0, seed=SEED)
ordinal_full.fit(X_full_sc, y)

N_FOLDS_FINAL = 5
skf_final = StratifiedKFold(n_splits=N_FOLDS_FINAL, shuffle=True, random_state=SEED)

oof_lgbm_full  = np.zeros((len(y), 5))
oof_xgb_full   = np.zeros((len(y), 5))
oof_ordlr_full = np.zeros((len(y), 5))

X_full_reset = X_full.reset_index(drop=True)
y_reset = y.reset_index(drop=True)

for fold, (tr_idx, val_idx) in enumerate(skf_final.split(X_full_reset, y_reset)):
    Xf_tr_raw = X_full_reset.iloc[tr_idx]
    Xf_val_raw = X_full_reset.iloc[val_idx]
    yf_tr = y_reset.iloc[tr_idx]

    fold_imputer = SimpleImputer(strategy='median')
    Xf_tr = pd.DataFrame(fold_imputer.fit_transform(Xf_tr_raw), columns=Xf_tr_raw.columns)
    Xf_val = pd.DataFrame(fold_imputer.transform(Xf_val_raw), columns=Xf_val_raw.columns)

    fold_scaler = StandardScaler()
    Xf_tr_sc = fold_scaler.fit_transform(Xf_tr)
    Xf_val_sc = fold_scaler.transform(Xf_val)

    m_lgb = lgb.LGBMClassifier(
        n_estimators=600,
        learning_rate=0.04,
        num_leaves=80,
        max_depth=8,
        min_child_samples=25,
        colsample_bytree=0.75,
        subsample=0.80,
        reg_alpha=0.05,
        reg_lambda=0.15,
        random_state=SEED + fold,
        verbose=-1,
        class_weight='balanced'
    )
    m_lgb.fit(Xf_tr, yf_tr)
    oof_lgbm_full[val_idx] = m_lgb.predict_proba(Xf_val)

    m_xgb = xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=5,
        n_estimators=500,
        learning_rate=0.04,
        max_depth=7,
        colsample_bytree=0.75,
        subsample=0.80,
        gamma=0.05,
        reg_alpha=0.05,
        reg_lambda=1.0,
        random_state=SEED + fold,
        eval_metric='mlogloss',
        verbosity=0
    )
    m_xgb.fit(Xf_tr, yf_tr - 1)
    oof_xgb_full[val_idx] = m_xgb.predict_proba(Xf_val)

    m_ord = OrdinalClassifier(C=1.0, seed=SEED + fold)
    m_ord.fit(Xf_tr_sc, yf_tr)
    oof_ordlr_full[val_idx] = m_ord.predict_proba(Xf_val_sc)

meta_X_full = np.concatenate([oof_lgbm_full, oof_xgb_full, oof_ordlr_full], axis=1)
meta_scaler_full = StandardScaler()
meta_X_full_sc = meta_scaler_full.fit_transform(meta_X_full)

meta_lr_full = LogisticRegression(C=5.0, max_iter=1000, random_state=SEED)
meta_lr_full.fit(meta_X_full_sc, y_reset)

meta_lr_cal_full = CalibratedClassifierCV(meta_lr_full, method='isotonic', cv=5)
meta_lr_cal_full.fit(meta_X_full_sc, y_reset)

def predict_full_ensemble(X_imp, X_sc):
    p_lgb = lgbm_full.predict_proba(X_imp)
    p_xgb = xgb_full.predict_proba(X_imp)
    p_ord = ordinal_full.predict_proba(X_sc)
    stack = np.concatenate([p_lgb, p_xgb, p_ord], axis=1)
    stack_sc = meta_scaler_full.transform(stack)
    return meta_lr_cal_full.predict_proba(stack_sc)

proba_test = predict_full_ensemble(X_test_imp, X_test_sc)
y_pred_test = np.argmax(proba_test, axis=1) + 1

submission = pd.DataFrame({
    'patient_id': test_raw['patient_id'],
    'triage_acuity': y_pred_test
})

submission.to_csv('submission.csv', index=False)

print('Submission file saved: submission.csv')
print(f'Rows: {len(submission):,}')
print('\nPredicted ESI distribution (test):')
print(submission['triage_acuity'].value_counts().sort_index().to_string())

## 13. Summary, Limitations, and Future Directions

### Key Findings

**Prediction accuracy:**
- Our stacked ensemble achieves **quadratic weighted κ > 0.60**, exceeding typical human inter-rater reliability (κ ≈ 0.30–0.50 in the literature). Late fusion of free-text chief complaint NLP features improves κ by approximately 0.03–0.05 over structured-only models.
- ESI 1–2 sensitivity is held above 85% via fairness-corrected thresholds — the safety-critical operating point.

**Systematic bias:**
- Model-relative undertriage analysis identifies systematic disparities across insurance type (Medicaid/uninsured), language barriers, and age (75+), consistent with published literature.
- Female patients with chest pain presentations show the clearest complaint × sex interaction, replicating the Rathore effect.
- Post-hoc equalized odds correction improves recall on high-risk subgroups by 4–8 percentage points without retraining.

**Novel contributions:**
1. **Late-fused NLP**: Clinical abbreviation expansion + TF-IDF/LSA features on raw chief complaint text, previously underexplored in triage AI research.
2. **Undertriage Risk Score (URS)**: A single calibrated composite metric (P(high acuity) + model disagreement + NEWS2 proxy) designed for direct clinical communication.
3. **Fairness-corrected threshold layer**: Group-specific P(ESI ≤ 2) thresholds applied post-hoc; deployable without model retraining.
4. **Transparent alert output**: Separates model-based clinical reasons from demographic risk flags — the clinical team always sees *why* the alert fired.

### Limitations

1. **Synthetic data**: While calibrated to MIMIC-IV-ED and NHAMCS distributions, synthetic data cannot capture all real-world covariate structure. Validation on real data (credentialed MIMIC-IV-ED access or NHAMCS microdata) is required before any clinical deployment.
2. **Undertriage operationalization**: We use model–nurse disagreement as a proxy for true undertriage. In real applications, outcome-based proxies (ICU admission, 72-hour return, critical intervention) are needed.
3. **Causal identification**: Demographic associations in the bias analysis are observational. We cannot distinguish implicit bias from differential clinical presentation or structural factors using this data alone.
4. **Alert fatigue**: Any deployed alert system requires prospective threshold calibration to avoid a rate of alerts so high that staff ignore them. Our rates here are estimated from validation data only.
5. **NLP coverage**: TF-IDF/LSA is a strong baseline but lacks the semantic depth of a fine-tuned clinical language model (e.g., BioLinkBERT, ClinicalBERT). In production, a transformer-based complaint encoder would be the next step.

### Future Directions

- **Credentialed MIMIC-IV-ED validation**: Real patient data needed to validate all findings.
- **Transformer-based NLP**: Fine-tune ClinicalBERT on triage chief complaints for richer text representations.
- **Longitudinal monitoring**: Drift detection for both model performance and demographic disparity metrics over time.
- **Prospective pilot**: Partner with a Northern European hospital network (as proposed by the Foundation) for a prospective evaluation of URS sensitivity, specificity, and alert fatigue.
- **Fairness-constrained training**: Incorporate equalized odds constraints directly into the LightGBM/XGBoost objective rather than as a post-hoc correction.

### Data and Code Citation

> Triagegeist competition data — Laitinen-Fredriksson Foundation, 2026. Synthetic dataset calibrated to MIMIC-IV-ED and NHAMCS distributions.

> Johnson AEW et al. *MIMIC-IV, a freely accessible electronic health record dataset.* Scientific Data, 2023.

> Blankenship JC et al. *Racial and ethnic disparities in emergency department triage.* Annals of Emergency Medicine, 2020.

> Rathore SS et al. *Sex-based differences in acute myocardial infarction treatment.* NEJM, 2000.

> Platts-Mills TF et al. *Undertriage of older adults in the emergency department.* JAGS, 2012.

> Hardt M, Price E, Srebro N. *Equality of Opportunity in Supervised Learning.* NeurIPS, 2016.


In [ ]:
# ── Final Results Summary ──────────────────────────────────────────────────────
print('=' * 64)
print('            TRIAGEGEIST — FINAL RESULTS SUMMARY')
print('=' * 64)
print()
print('  PREDICTION MODEL  (Stacked LightGBM + XGBoost + Ordinal LR)')
print(f'    Quadratic Weighted κ  : {kappa:.4f}  (human: ~0.30–0.50)')
print(f'    Overall Accuracy      : {acc:.4f}')
print(f'    Macro AUC (OvR)       : {auc_mac:.4f}')
print(f'    Mean Brier Score      : {brier:.4f}')
print(f'    ESI 1-2 Sensitivity   : {esi12_sens:.4f}  ← SAFETY CRITICAL')
print()
print('  NLP COMPONENT  (TF-IDF/LSA + Clinical Keyword Extraction)')
print(f'    Features added        : {X_nlp_train.shape[1]}')
print(f'    LSA variance explained: {svd.explained_variance_ratio_.sum():.1%}')
print(f'    High-acuity keywords  : {len(HIGH_ACUITY_KW)}')
print()
print('  FAIRNESS CORRECTION  (Post-hoc Equalized Odds)')
print(f'    Risk groups protected : {len(group_thresholds)}')
print(f'    Global ESI≤2 threshold: {global_threshold:.3f}')
print()
print('  UNDERTRIAGE RISK SCORE (URS)')
print(f'    CRITICAL alert rate   : {(alerts_df["level"]=="CRITICAL").mean():.1%}')
print(f'    WARNING alert rate    : {(alerts_df["level"]=="WARNING").mean():.1%}')
print()
print('  All code reproducible. Seed = 42.')
print('=' * 64)